<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part3_transformers/10_vlms_as_annotators/notebooks/10_vlms_as_annotators_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 10: Vision-Language Models as Annotators · Solutions

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a short comment. There is usually more than one good way to solve an exercise, and your numbers may differ a little from run to run, so read the comments as "what to look for" rather than exact results.

A vision-language model (VLM) is a language model that can look at images. You can ask it what a
picture shows, give it a coding sheet to fill in, or have it describe a scene in words that a text
pipeline can analyze. Small open VLMs now run on a free Colab GPU, which makes them a realistic
annotation tool.

This notebook treats a VLM the way you would treat a new research assistant: give it your
codebook, check its work against yours, look closely at its mistakes, and only trust it where it
has earned that trust.

In this notebook you will:

- load a small open VLM and ask it about a few images
- turn the module 04 codebook into a prompt that returns structured JSON, code your 40 test images
  with it and validate the result against your own codes, variable by variable
- catch the model inventing things that aren't in the image
- check how much the results depend on the prompt wording and the image resolution
- try two ways of scaling up: describe-then-analyze, and distilling the VLM's labels into a fast
  classifier on CLIP embeddings

**Data.** The protest and King's Day photos with your codes from module 04 (all four variables).

**Model.** `Qwen/Qwen3-VL-2B-Instruct` (Apache 2.0, about 2 billion parameters, no account needed).

**Runtime.** You need a GPU: *Runtime → Change runtime type → T4 GPU*. The whole notebook takes
about half an hour on a T4, most of it in sections 5 and 6.

## 0. Setup

Qwen3-VL needs transformers 4.57 or newer. The first cell installs it only if Colab's version is
older.

In [ ]:
!pip install -q "transformers>=4.57"

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score
from sklearn.pipeline import make_pipeline

SEED = 42
np.random.seed(SEED)

HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}
BASE = "https://raw.githubusercontent.com/denisejroth/pixels-frames-transformers/main/shared/"
for helper in ["course_data.py", "course_torch.py"]:
    if not Path(helper).exists():
        response = requests.get(BASE + helper, headers=HEADERS, timeout=30)
        response.raise_for_status()  # stop if the download failed, rather than saving an error page
        Path(helper).write_text(response.text)
import course_data
import course_torch
from course_torch import DEVICE, load_rgb

print("Running on:", DEVICE)
ON_GPU = DEVICE == "cuda"

def show_grid(paths, labels=None, n_cols=8, size=1.6):
    """Show images in a grid, with an optional label under each."""
    paths = list(paths)
    labels = list(labels) if labels is not None else [""] * len(paths)
    n_rows = max(1, int(np.ceil(len(paths) / n_cols)))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * size, n_rows * size * 1.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, path, label in zip(axes.flat, paths, labels):
        ax.imshow(load_rgb(path, 300))
        ax.set_title(str(label)[:28], fontsize=7)
    plt.tight_layout()
    plt.show()

VARIABLES = ["protest", "crowd", "police", "signs"]
train_table, test_table = course_data.protest_train_test()
my_codes = pd.read_csv("my_codes.csv")
test_table = test_table.drop(columns=["protest"]).merge(my_codes[["title", *VARIABLES]], on="title")
print(test_table[VARIABLES].mean().round(2).rename("share coded yes by you"))

## 1. How a VLM works, and a first conversation

Most open VLMs have three parts:

1. a **vision encoder**, a vision transformer like the ones in modules 08 and 09, which turns the
   image into a grid of patch tokens;
2. a **connector**, a small network that maps those tokens into the language model's embedding
   space (Qwen3-VL also merges each 2 × 2 block of neighboring patches into one token, so a
   768 × 768 image becomes a few hundred tokens);
3. a **language model**, which reads the image tokens as if they were words, followed by your
   question, and writes an answer one token at a time.

The language model is where the knowledge, the instruction following and the hallucinations come
from. `course_torch.load_vlm()` loads the model once; `course_torch.ask_vlm(images, prompt)`
shows it one or more images with a prompt and returns the answer. Two choices are made for you:

- **Precision.** The T4 has no support for bfloat16, the format these weights are stored in, and
  float16 can overflow in these models and produce empty or garbled answers. So on a T4 the model
  runs in float32, which takes about 9 GB of the T4's 15 GB and is slower but safe. On newer GPUs
  it uses bfloat16.
- **Greedy decoding.** The model always picks the most likely next token, so the same image and
  prompt give the same answer every time. For annotation you want that; for creative writing you
  wouldn't.

In [ ]:
VLM = course_torch.VLM_MODEL
processor, vlm = course_torch.load_vlm(VLM)
print(VLM, f"{sum(p.numel() for p in vlm.parameters()) / 1e9:.1f} billion parameters, {next(vlm.parameters()).dtype}")

example = test_table["local_path"].iloc[0]
show_grid([example], n_cols=1, size=4)
for question in ["Describe this image in two sentences.",
                 "How many people can you see? Answer with a number.",
                 "Is there a police officer in this image? Answer yes or no."]:
    start = time.time()
    answer = course_torch.ask_vlm(example, question)
    print(f"Q: {question}\nA: {answer}  ({time.time() - start:.1f}s)\n")

> **✏️ Exercise 1**
>
> Pick three test images that you found hard to code in module 04 (or any three). For each, ask the
> model one open question ("What is happening in this image?") and one question about a variable in
> your codebook. Where does the model see what you saw, and where not?

In [ ]:
# ✅ Solution
for path in test_table["local_path"].sample(3, random_state=SEED):
    show_grid([path], n_cols=1, size=3)
    print("What is happening?", course_torch.ask_vlm(path, "What is happening in this image? Answer in two sentences."))
    print("Signs?", course_torch.ask_vlm(path, "Is anyone carrying or displaying a sign, banner, placard or flag with a message or symbol? Answer yes or no."))
    print()

# Typical observations: the descriptions are fluent and mostly right about the overall scene, and
# often confident about details that are hard to see. Yes/no answers follow the words of the
# question closely, so a question that mirrors your codebook definition works better than a
# vague one. Write down any disagreement: you will see more of them in section 2.

## 2. Your codebook as a prompt

A good prompt for annotation looks like a good codebook: the unit of analysis, each variable with
its definition and the rules for hard cases, and an exact output format. Asking for **JSON** makes
the answers easy to read into a table and makes it obvious when the model doesn't follow the
format. The definitions below are the ones from module 04.

In [ ]:
CODEBOOK_PROMPT = """You are coding photos for a study of protest. Code only what is visible in the photo.

Variables (1 = yes, 0 = no):
- protest: people engaged in a protest: marching, gathering with signs or banners, a blockade or sit-in, chanting. A single person holding a protest sign counts. Code 0 for only objects (a poster on a wall, an empty square), police without protesters, or a crowd with no visible sign of protest (a festival, a market, a celebration).
- crowd: roughly ten or more people are visible, including partly visible people in the background.
- police: police officers, police vehicles or barriers marked as police are visible. Security guards and stewards don't count.
- signs: at least one sign, banner, placard or flag with a message or symbol is carried or displayed by people. Ordinary signs such as street names, shop signs and traffic signs don't count.

Answer with JSON only, in exactly this format:
{"protest": 0, "crowd": 0, "police": 0, "signs": 0}"""

def code_images(paths, prompt=CODEBOOK_PROMPT, max_side=768, variables=VARIABLES):
    """Code each image with the VLM. Returns one row per image, with the raw answer kept."""
    rows = []
    for path in paths:
        answer = course_torch.ask_vlm(path, prompt, max_new_tokens=60, max_side=max_side)
        record = course_torch.parse_json(answer) or {}
        row = {"local_path": path, "raw_answer": answer, "parsed": bool(record)}
        for v in variables:
            value = record.get(v)
            row[v] = int(value) if value in (0, 1, "0", "1", True, False) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

start = time.time()
vlm_codes = code_images(test_table["local_path"])
print(f"coded {len(vlm_codes)} images in {time.time() - start:.0f}s; answers that weren't valid JSON: {(~vlm_codes['parsed']).sum()}")
vlm_codes.head()

Never drop failed answers silently. If the model refused or wrote something else for some images,
those images are missing *not at random*: they are often the hard ones. Count them, look at them,
and report them. Below, a missing code counts as a disagreement.

In [ ]:
def validate(human, machine, variables=VARIABLES):
    rows = []
    for v in variables:
        m = machine[v].fillna(-1).astype(int)
        rows.append({"variable": v, "share yes (you)": human[v].mean(), "share yes (VLM)": (m == 1).mean(),
                     "agreement": (human[v].to_numpy() == m.to_numpy()).mean(),
                     "balanced accuracy": balanced_accuracy_score(human[v], m.clip(lower=0)),
                     "kappa": cohen_kappa_score(human[v], m.clip(lower=0)),
                     "missing": machine[v].isna().sum()})
    return pd.DataFrame(rows).set_index("variable").round(2)

validation = validate(test_table.reset_index(drop=True), vlm_codes)
validation

In [ ]:
print(course_data.log_score(f"VLM zero-shot ({VLM.split('/')[-1]}), codebook prompt", test_table["protest"],
                            vlm_codes["protest"].fillna(0).astype(int), module="10"))

Read the table variable by variable. Kappa corrects agreement for chance, so a variable that is
rare (say, police in 5 of 40 images) can have high agreement and low kappa. Compare the shares too:
a model that says "yes" much more or less often than you do will bias any prevalence you estimate
with it, even if its accuracy looks fine (module 07 showed what happens when that bias differs
between groups).

> **✏️ Exercise 2**
>
> Look at the disagreements. For the variable with the lowest kappa, show the images where the model
> and you disagree, with your code, the model's code and its raw answer. For each one, decide who is
> right. Are some of the "errors" actually your mistakes, or gaps in the codebook?

In [ ]:
# ✅ Solution
worst = validation["kappa"].idxmin()
merged = test_table.reset_index(drop=True).join(vlm_codes[[worst, "raw_answer"]].add_prefix("vlm_"))
disagree = merged[merged[worst] != merged[f"vlm_{worst}"].fillna(-1)]
print(f"{worst}: {len(disagree)} disagreements")
show_grid(disagree["local_path"], [f"you {a} / VLM {b}" for a, b in zip(disagree[worst], disagree[f"vlm_{worst}"])],
          n_cols=6, size=2)
for _, row in disagree.head(6).iterrows():
    print(Path(row["local_path"]).name, "->", row["vlm_raw_answer"][:120])

# Expect a mix. Some disagreements are clear model errors (a festival flag read as a protest sign,
# a crowd in the distance missed). Some are borderline cases the codebook doesn't settle (a few
# people with one small flag). And some are your own slips: coders tire, and module 04's recoding
# showed you are not perfectly stable either. The human codes are the benchmark, not the truth;
# disagreements are a reason to look again, and possibly to refine the codebook.

## 3. Hallucination: describing what isn't there

A language model writes the most plausible continuation, and a plausible description of a protest
includes police, signs and chanting whether or not they are in the picture. This is
**hallucination** (Rohrbach et al., 2018, measured it for objects in image captions). It is worst
with **leading questions** that presuppose something: "What are the police officers doing?" invites
the model to describe police.

A simple test: take images where *you* coded no police, and ask a neutral question and a leading
one.

In [ ]:
no_police = test_table[test_table["police"] == 0]["local_path"].head(8 if ON_GPU else 3).tolist()
NEUTRAL = "Is there a police officer in this image? Answer yes or no."
rows = []
for path in no_police:
    rows.append({"image": Path(path).name,
                 "neutral question": course_torch.ask_vlm(path, NEUTRAL, max_new_tokens=5)})
neutral_table = pd.DataFrame(rows)
neutral_table

> **✏️ Exercise 3**
>
> Ask the same images a leading question: `"Describe what the police officers in this image are
> doing, in one sentence."` In how many answers does the model describe police that you didn't see
> (versus saying there are none)? Look at one or two of those images again: did you miss something,
> or did the model invent it?

In [ ]:
# ✅ Solution
LEADING = "Describe what the police officers in this image are doing, in one sentence."
neutral_table["leading question"] = [course_torch.ask_vlm(p, LEADING, max_new_tokens=40) for p in no_police]
denies = neutral_table["leading question"].str.contains(r"\bno police|no officers|there are no|not see|cannot see|can't see|don't see", case=False)
print(f"the model described police in {(~denies).sum()} of {len(neutral_table)} images you coded without police")
pd.set_option("display.max_colwidth", 120)
neutral_table

# Small models often go along with the presupposition in some of these images and describe officers
# who aren't there, even when they answered "no" to the neutral question. This is why annotation
# prompts should ask neutral questions with explicit "0 = not visible" options, and why open
# descriptions from a VLM should be treated as claims to check, not as observations. (The simple
# word search above can miss some ways of saying "there are none", so read the answers.)

## 4. Sensitivity checks

An annotation is only useful if it doesn't change with arbitrary choices. Two checks are cheap and
should be routine: rewording the prompt, and changing the image resolution. Here is a shorter
prompt with the same variables, in a different order.

In [ ]:
SHORT_PROMPT = """Look at the photo. Answer with JSON only: {"signs": 0, "police": 0, "crowd": 0, "protest": 0}.
Use 1 if the photo shows: signs (a sign, banner, placard or flag with a message carried by people);
police (police officers or vehicles); crowd (about ten or more people); protest (people protesting or demonstrating)."""

vlm_short = code_images(test_table["local_path"], prompt=SHORT_PROMPT)
compare = pd.DataFrame({
    "agreement between prompts": [(vlm_codes[v].fillna(-1) == vlm_short[v].fillna(-1)).mean() for v in VARIABLES],
    "kappa with you, codebook prompt": validation["kappa"].to_numpy(),
    "kappa with you, short prompt": validate(test_table.reset_index(drop=True), vlm_short)["kappa"].to_numpy(),
}, index=VARIABLES).round(2)
compare

If two reasonable prompts disagree on, say, 15% of images, then any finding that depends on those
images is fragile. Report this agreement the way you would report agreement between two human
coders.

> **✏️ Exercise 4**
>
> Now vary the resolution. `ask_vlm` shrinks each image so its longest side is at most `max_side`
> pixels (768 by default). Code the test images again with `max_side=336`, which gives the model about
> a fifth as many image tokens. How much does agreement with your codes change, and for which
> variables? Which variables would you expect to suffer most?

In [ ]:
# ✅ Solution
vlm_small = code_images(test_table["local_path"], max_side=336)
resolution = pd.DataFrame({
    "kappa, 768 px": validation["kappa"].to_numpy(),
    "kappa, 336 px": validate(test_table.reset_index(drop=True), vlm_small)["kappa"].to_numpy(),
    "agreement between resolutions": [(vlm_codes[v].fillna(-1) == vlm_small[v].fillna(-1)).mean() for v in VARIABLES],
}, index=VARIABLES).round(2)
resolution

# Variables that depend on small details suffer most: police (a few officers at the edge of the
# frame) and signs (text on a placard), and crowd when people are far away. Protest, which depends
# on the scene as a whole, is often more robust. Lower resolution is faster, so the trade-off is
# worth measuring for your own variables before coding a large collection.

## 5. Describe, then analyze the text

Another way to use a VLM is to have it describe each image in words and then analyze the
descriptions with text methods, for example everything from Bags, Vectors & Transformers. Here we
describe a sample of the training images and the test images, train a TF-IDF classifier on the
descriptions of the training images (with their category labels) and score it on your codes.

Something always gets lost on the way: whatever the model doesn't mention can't be analyzed, and
whatever it invents can.

In [ ]:
DESCRIBE = "Describe this photo in two sentences: who is in it, what they are doing, and the setting."
N_DESCRIBE = 120 if ON_GPU else 20
describe_train = pd.concat([g.sample(min(len(g), N_DESCRIBE // 2), random_state=SEED)
                            for _, g in train_table.groupby("category_label")]).reset_index(drop=True)

start = time.time()
describe_train["description"] = [course_torch.ask_vlm(p, DESCRIBE, max_new_tokens=80) for p in describe_train["local_path"]]
test_descriptions = [course_torch.ask_vlm(p, DESCRIBE, max_new_tokens=80) for p in test_table["local_path"]]
print(f"{len(describe_train) + len(test_table)} descriptions in {time.time() - start:.0f}s")
describe_train[["description", "category_label"]].head()

In [ ]:
text_model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                           LogisticRegression(class_weight="balanced", max_iter=2000))
text_model.fit(describe_train["description"], describe_train["category_label"])
print(course_data.log_score("VLM descriptions + TF-IDF classifier", test_table["protest"],
                            text_model.predict(test_descriptions), module="10"))

vectorizer, logit = text_model.named_steps.values()
weights = pd.Series(logit.coef_[0], index=vectorizer.get_feature_names_out()).sort_values()
print("words pointing to King's Day:", ", ".join(weights.head(10).index))
print("words pointing to protest:   ", ", ".join(weights.tail(10).index))

The word lists are a nice by-product: they show what the model *mentions* when it sees each kind of
event. That is the strength of describe-then-analyze, its transparency. Its weaknesses are the cost
(writing two sentences per image is much slower than producing a code) and the loss of anything the
model doesn't put into words.

## 6. Distillation: VLM labels, fast classifier

Coding a million images with a VLM is slow. A common alternative is **distillation**: code a few
hundred images with the VLM, use those codes as training labels for a fast classifier on CLIP
embeddings (module 09), and run that classifier on the whole collection. The fast classifier
inherits the VLM's errors and adds its own, so it has to be validated on human codes like anything
else.

Here we code the training images we described in section 5 with the codebook prompt, and train the
CLIP probe on those VLM labels instead of the Commons categories.

In [ ]:
start = time.time()
train_vlm = code_images(describe_train["local_path"], variables=["protest"])
print(f"coded {len(train_vlm)} training images in {time.time() - start:.0f}s")

usable = train_vlm["protest"].notna().to_numpy()
clip_train = course_torch.clip_images(describe_train["local_path"][usable])
clip_test = course_torch.clip_images(test_table["local_path"])
distilled = LogisticRegression(class_weight="balanced", max_iter=2000).fit(clip_train, train_vlm["protest"][usable].astype(int))
print(course_data.log_score("CLIP probe trained on VLM labels", test_table["protest"], distilled.predict(clip_test), module="10"))

> **✏️ Exercise 5**
>
> On the same training images there are now two sets of labels: the Commons categories and the VLM's
> codes. Compare them in a cross-table. Then train the same CLIP probe on the Commons labels of these
> images, score it on your codes, and log it as `"CLIP probe trained on category labels (same images)"`.
> Which labels made the better teacher?

In [ ]:
# ✅ Solution
both = describe_train.assign(vlm_protest=train_vlm["protest"])[usable]
print(pd.crosstab(both["category_label"], both["vlm_protest"], rownames=["Commons category"], colnames=["VLM code"]))

category_probe = LogisticRegression(class_weight="balanced", max_iter=2000).fit(clip_train, both["category_label"])
print(course_data.log_score("CLIP probe trained on category labels (same images)", test_table["protest"],
                            category_probe.predict(clip_test), module="10"))

# The two label sources disagree on a noticeable share of images: some photos in protest
# categories show no protest by the codebook's definition (an empty square, a portrait of a
# speaker), and the VLM makes its own mistakes. Which teacher wins depends on which kind of noise
# is closer to your codes. Because the VLM followed *your codebook*, its labels often match your
# definition better than the category labels do, even when they are noisier image by image.

In [ ]:
course_data.keep_my_files()
course_data.show_scores()

> **✏️ Exercise 6**
>
> You used a VLM to code 20,000 images for a paper. Write the list of things your methods section
> should report so that a reviewer can judge, and someone else can repeat, your measurement.

**✅ One possible answer**

- **The model, exactly:** name, size, version or commit of the weights, the library version, the
  precision (float32, bfloat16), and the decoding settings (greedy, or temperature and seed).
- **The prompt, in full**, in an appendix, including the codebook definitions and the output
  format, plus how answers that didn't follow the format were handled and how many there were.
- **Image handling:** resolution (`max_side`), cropping, and anything removed beforehand.
- **Validation:** a random sample coded by humans (with their own reliability), and per variable the
  agreement, kappa or F1, and the share of "yes" from humans versus the model. Report errors per
  relevant group or context if you compare groups.
- **Sensitivity:** agreement across reasonable prompt wordings and resolutions, and whether the main
  findings hold under the alternatives.
- **Hallucination checks** for variables that the model could plausibly invent.
- **If you distilled:** the size and composition of the VLM-labeled training set and the validation
  of the final classifier on human codes.
- **Openness:** whether the weights are open and versioned (so the result can be reproduced next
  year), unlike a commercial API whose model can change without notice.